# High-Level Backtesting of the Turtle Soup Strategy

In [1]:
# imports
import time
import pandas as pd
from nautilus_trader.backtest.config import BacktestVenueConfig, BacktestDataConfig, BacktestRunConfig
from nautilus_trader.backtest.engine import BacktestResult, BacktestEngine, BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.common.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType, Bar, Venue, InstrumentId
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.config import DataCatalogConfig
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading.config import ImportableStrategyConfig
import sys
from pathlib import Path

from core.enums import MoneyManagementType

In [2]:
#sys.path.append(str(Path.cwd().parent))

catalog = ParquetDataCatalog("../catalog")

start_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-01"))
end_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-19"))

instrument = TestInstrumentProvider.es_future(2025, 12)
instrument_id = instrument.id.value

# Configure backtesting
venue = BacktestVenueConfig(
    name="GLBX",
    oms_type=OmsType.NETTING,
    account_type="MARGIN",
    base_currency="USDT",
    starting_balances=["10_000 USDT"],
)

# Configure a catalog for a live system
catalog_cfg = DataCatalogConfig(
    path=str(catalog.path),
    fs_protocol="file",
    name="local"
)

base_bar_type = BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL")
data = BacktestDataConfig(
    catalog_path=str(catalog.path),
    catalog_fs_protocol="file",
    data_cls=Bar,
    bar_types=[base_bar_type],
    instrument_id=instrument_id,
    start_time=start_ns,
    end_time=end_ns
)

engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategy",
            config_path="strategies.turtle_soup.turtle_soup_strategy:TurtleSoupStrategyConfig",
            config={
                "liquidity_pool_bar_type": BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "liquidity_pool_upper_period_window": 3,
                "liquidity_pool_lower_period_window": 3,

                "turtle_soup_bar_type": BarType.from_str(f"{instrument_id}-1-HOUR-LAST-INTERNAL@1-MINUTE-EXTERNAL"),
                "turtle_soup_bars_count": 4,

                "risk_reward_ratio": 2.0,

                "money_management_type": MoneyManagementType.FIXED_RISK_PERCENT,
                "fixed_lot": 0.5,
                "fixed_risk_percent": 1,

                "instrument_id": instrument_id,
                "base_bar_type": base_bar_type,
                "is_backtest": True,
            },
        ),
    ],
    logging=LoggingConfig(log_level="ERROR"),
    catalogs=[catalog_cfg]
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

node = BacktestNode(configs=[config])

# run backtesting
elapsed_start = time.perf_counter()
# Runs one or many configs synchronously
results: list[BacktestResult] = node.run()
elapsed_end = time.perf_counter()

print(f"Elapsed time: {elapsed_end - elapsed_start:.6f} seconds")

Elapsed time: 0.739526 seconds


In [3]:
results

[BacktestResult(trader_id='BACKTESTER-001', machine_id='Yuriis-MacBook-Pro.local', run_config_id='22f30a959a9fa762ed2f2f9e476bee64a2b56fb9fee8f9fb3dceee6c6171c370', instance_id='f36b0020-a41e-4cca-bf07-a9f1c6caf7b8', run_id='ec7c4570-921d-44e0-845d-18dc34fd4d4e', run_started=1761062787716051000, run_finished=1761062788278497000, backtest_start=1759276800000000000, backtest_end=1760734800000000000, elapsed_time=1458000.0, iterations=22019, total_events=0, total_orders=0, total_positions=0, stats_pnls={'USDT': {'PnL (total)': 0.0, 'PnL% (total)': 0.0, 'Max Winner': 0.0, 'Avg Winner': 0.0, 'Min Winner': 0.0, 'Min Loser': 0.0, 'Avg Loser': 0.0, 'Max Loser': 0.0, 'Expectancy': 0.0, 'Win Rate': 0.0}}, stats_returns={'Returns Volatility (252 days)': nan, 'Average (Return)': nan, 'Average Loss (Return)': nan, 'Average Win (Return)': nan, 'Sharpe Ratio (252 days)': nan, 'Sortino Ratio (252 days)': nan, 'Profit Factor': nan, 'Risk Return Ratio': nan})]